# 🎬 HYPERFLIX: Pipeline de Procesamiento Distribuido Big Data
### Arquitectura: Data Lake (DBFS) ➔ Apache Spark Engine ➔ Curated Layer ➔ BI
**Asignatura:** Bases de Datos y Almacenamiento Masivo (Octavo Semestre)  
**Entorno:** Databricks Community Edition (Apache Spark 3.5+ en clúster Linux)  
**Autor:** Emerson Iles — Ingeniería de Sistemas (ITP)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print('⚡ Versión activa de Apache Spark en Databricks:', spark.version)
print('💻 Clúster conectado:', spark.sparkContext.master)

## 📥 2. Ingesta desde Zona Raw a Spark DataFrame
Telemetría de streaming y catálogo audiovisual de HYPERFLIX.

In [ ]:
datos_streaming = [
    ('EVT-001', 'MOV-0101', 'El Padrino', 'Crimen, Drama', 175, 160, 'SMART_TV', 'AndroidTV', 4500, 22.5, 0),
    ('EVT-002', 'MOV-0102', 'Inception', 'Acción, Ciencia Ficción', 148, 145, 'LAPTOP', 'Windows', 5800, 18.0, 1),
    ('EVT-003', 'MOV-0103', 'Interstellar', 'Ciencia Ficción, Drama', 169, 45, 'SMARTPHONE', 'iOS', 3200, 45.0, 3),
    ('EVT-004', 'MOV-0104', 'The Dark Knight', 'Acción, Crimen, Drama', 152, 150, 'SMART_TV', 'Tizen', 6200, 15.0, 0),
    ('EVT-005', 'MOV-0105', 'Pulp Fiction', 'Crimen, Comedia', 154, 154, 'TABLET', 'iPadOS', 4100, 28.0, 0),
    ('EVT-006', 'MOV-0106', 'Matrix', 'Acción, Ciencia Ficción', 136, 130, 'SMART_TV', 'WebOS', 5400, 19.5, 0),
    ('EVT-007', 'MOV-0107', 'Gladiador', 'Acción, Aventura, Drama', 155, 60, 'SMARTPHONE', 'Android', 2900, 55.0, 4),
    ('EVT-008', 'MOV-0108', 'Parasite', 'Drama, Suspenso', 132, 130, 'LAPTOP', 'macOS', 4900, 21.0, 1),
    ('EVT-009', 'MOV-0109', 'Avatar', 'Acción, Aventura, Fantasía', 162, 160, 'SMART_TV', 'AndroidTV', 6800, 14.0, 0),
    ('EVT-010', 'MOV-0110', 'Coco', 'Animación, Aventura, Familiar', 105, 105, 'SMART_TV', 'Roku', 4200, 20.0, 0)
]

columnas = [
    'evento_id', 'codigo_pelicula', 'titulo', 'generos_str', 'duracion_min',
    'minutos_vistos', 'tipo_dispositivo', 'sistema_operativo', 'bitrate_kbps',
    'latencia_ms', 'buffering_veces'
]

df_raw = spark.createDataFrame(datos_streaming, columnas)
display(df_raw)

## 🔄 3. Transformación 1: explode() (Convertir Lista en Varias Filas)

In [ ]:
df_con_array = df_raw.withColumn('array_generos', F.split(F.col('generos_str'), r'\s*,\s*'))
df_exploded = df_con_array.withColumn('genero_individual', F.explode(F.col('array_generos')))
display(df_exploded.select('evento_id', 'titulo', 'genero_individual'))

## ⚙️ 4. Transformación 2: Limpieza, Enriquecimiento y QoS (withColumn)

In [ ]:
df_enriquecido = (
    df_exploded
    .withColumn('completitud_pct', F.round((F.col('minutos_vistos') / F.col('duracion_min')) * 100, 2))
    .withColumn('abandono_prematuro', F.when(F.col('completitud_pct') < 20.0, F.lit(True)).otherwise(F.lit(False)))
    .withColumn('score_qos', F.round(F.greatest(F.lit(0.0), F.lit(100.0) - (F.col('latencia_ms') * 0.4) - (F.col('buffering_veces') * 10.0)), 1))
    .withColumn('procesado_por', F.lit('Databricks Cloud Engine'))
)
display(df_enriquecido.select('titulo', 'genero_individual', 'completitud_pct', 'score_qos'))

## 📊 5. Transformación 3: Agregaciones Analíticas (groupBy + agg)

In [ ]:
df_kpis_genero = (
    df_enriquecido
    .groupBy('genero_individual')
    .agg(
        F.count('evento_id').alias('total_reproducciones'),
        F.round(F.avg('completitud_pct'), 1).alias('completitud_promedio_pct'),
        F.round(F.avg('score_qos'), 1).alias('qos_promedio'),
        F.round(F.sum('minutos_vistos') / 60, 2).alias('horas_totales_streaming'),
        F.sum(F.when(F.col('abandono_prematuro') == True, 1).otherwise(0)).alias('total_abandonos')
    )
    .orderBy(F.col('total_reproducciones').desc())
)
display(df_kpis_genero)

## 💾 6. Exportación Nativa a DBFS y Delta Lake

In [ ]:
ruta_dbfs_parquet = 'dbfs:/FileStore/hyperflix/curated/kpis_genero_parquet'
ruta_dbfs_delta = 'dbfs:/FileStore/hyperflix/curated/kpis_genero_delta'

df_kpis_genero.write.mode('overwrite').parquet(ruta_dbfs_parquet)
df_kpis_genero.write.format('delta').mode('overwrite').save(ruta_dbfs_delta)
print('✅ Tablas guardadas en DBFS nativamente con Apache Spark.')